%md
#### Goal
- To create a dataobject to have all information about a CDD Case  for all W&R Sourcesystems

#### Authors
- Abhishek.R.Jaiswal@rabobank.com
- Mahalakshmi.Vengateswaran@rabobank.com
- Prasad.gadidala@rabobank.com
- Prajit.tatari@rabobank.com

##### Flow of Logic
- Reading GCOB,Legacy2,GCDS,GIC,KN1 Caseservice dataobjects from GDP
- Fetching the Cases details of GCOB Clients from party_case_client_details
- Fetching the Cases details of Legacy2 Clients from Legacy2_case_client_details
- Fetching the Cases details of GIC Clients from KN1 Dataobjects
- Combining all the case related information from different sourcesystems
- Loading the final dataobject to SA RADAR Storage account

##### Version & Changes
|     Developer |Date	   | PBI/Bug No |	Changes done | Version
|----------|----------|----------|----------|----------|
| Abhishek Jaiswal	 |9-July-2025 |13019364 |Changes CaseStatusName for KN1	| 1	
| Prajit Tatari	 |29-Aug-2025 |13538827 |Adding RiskModelName  | 2
| Mahalakshmi V | 20-Nov-2025 | 14165735 | Made notebook compatible for Historical Load | 2
| Prajit Tatari	 |11-Feb-2026 |15058674 |Adding MDM RANZ data from GDP  | 2
|Rhea Gupta | 30-Mar-2026 |15580816 | Included RANZ in version 3 | 3
| Mahalakshmi V | 20-May-2026 | 16071492 | Included RANZ V4 |4

#####READING FILES FROM GDP

In [0]:
import os
import pandas as pd
from datetime import datetime, timedelta

In [0]:
from RadarUtils import *

In [0]:
app_reg_app_id = os.environ['APP_REG_APP_ID']
ReadStorage = os.environ['GDP_STORAGE_NAME']
TenantId = os.environ['TENANT_ID']
GIC_ReadStorage = os.environ['GDP_SA_STORAGE_NAME']
RANZ_ReadStorage = os.environ['AU_GDP_Defined_Storage_Account']
environment=os.environ['ENV']
radar_datamodel_version_number=3

In [0]:
dbutils.widgets.text("Load_Date", "")
dbutils.widgets.text("RunType", "daily")  # default is daily

Load_Date = dbutils.widgets.get("Load_Date")
RunType = dbutils.widgets.get("RunType").lower()
if Load_Date:
    RunType="historical"
    
if not Load_Date:
    Load_Date = datetime.today().strftime('%Y%m%d')

print(f"Running for Load Date: {Load_Date}")

In [0]:
party_CDDcase_dataobject='Party_CDDCase'

In [0]:
SARADAR='saradar'+environment

In [0]:
authenticate_storage_account(ReadStorage)
authenticate_storage_account(GIC_ReadStorage)
authenticate_storage_account(RANZ_ReadStorage)
authenticate_storage_account(SARADAR)

In [0]:
# Derive the date for which data has to be processed from GDP
load_dts = f"EDL_LOAD_DTS={Load_Date}*"
print (load_dts)

In [0]:
# List of datasets from GDP
load_df = [
'party_case_client_details'
]

# Create TempView for each loading table
for Object in load_df:
    Read_GDP_Defined_DataObjects(Source='GCOB', Dataobject=Object, path_prefix='CaseService',Load_Date=Load_Date)

In [0]:
df_Party_SystemIdentifier=spark.read.parquet(f'abfss://radardatamodel@{SARADAR}.dfs.core.windows.net/Party_SystemIdentifier/3/data/{load_dts}/*.parquet')

In [0]:
# List of datasets from GDP
load_df = [
'Legacy2_case_client_details',
'Legacy2_risk']

# Create TempView for each loading table
for Object in load_df:
    Read_GDP_Defined_DataObjects(Source='Legacy2', Dataobject=Object,path_prefix='Legacy2',Load_Date=Load_Date)

In [0]:
# print list of strings for loading spark dfs from GDP
load_df = pd.DataFrame({'GDPname':[
'pessoa'
]})
#gic_load_dts
# Create TempView for each loading table
for index, row in load_df.iterrows():
    Read_GDP_Defined_DataObjects(Source='GIC', Dataobject=row.GDPname,Load_Date=Load_Date)

In [0]:
# print list of strings for loading spark dfs from GDP
load_df = pd.DataFrame({'GDPname':[
'ADKYC_CONTRAPARTES'
,'ADRBB_CONTRAPARTES_GRAM'
,'ADKYC_CONTRAPARTES_COMPL'
,'ADKYC_CONTRAPARTES_FASES'
,'ADKYC_SITUACOES'
]})

# Create TempView for each loading table
for index, row in load_df.iterrows():
    Read_GDP_Defined_DataObjects(Source='KN1', Dataobject=row.GDPname,Load_Date=Load_Date)

In [0]:
# List of dataobjects from GDP
load_df =[
# 'Party_RiskModelInstanceQuestionAnswers'
# , 'Party_RiskModelCategories'
'Party_RiskModelInstance'
# , 'party_WRCDDModel'
]

# Create TempView for each loading table
for Object in load_df:
    Read_GDP_Defined_DataObjects(Source='GCOB', Dataobject=Object, path_prefix='RISKMODEL',Load_Date=Load_Date)

In [0]:
df_Party_KN1Cases = spark.read.parquet(f"abfss://radardatamodel@{SARADAR}.dfs.core.windows.net/Party_KN1Cases/3/data/{load_dts}/*.parquet")
df_Party_KN1Cases.createOrReplaceTempView("KN1Cases")

In [0]:
load_df = [
    'c_b_party_xref'
,'c_lkp_cdd_risk_rating_xref'
,'c_b_due_diligence_xref'
,'c_b_contract_xref'
,'c_b_contr_rol_party_xref'
]
load_ranz_v4_rdm_tables(load_df, Load_Date)

#####TRANSFORMATION TO GET Party_CDDCase OBJECT 

In [0]:
#Static Other Risk Level
Risk_list = [0,1,2,3,4,9999]
Risk_Description = ['Risk not assigned', 'Unacceptable',  'High', 'Medium','Low','Unknown']
# create pyspark dataframe from lists
df_gic_static_RiskLevel = spark.createDataFrame(zip(Risk_list, Risk_Description), ['GIC_Risk_Id', 'GIC_Risk_Description'])
df_gic_static_RiskLevel.createOrReplaceTempView('gic_static_Other_RiskLevel')

#Creating Static view for Risk Level
Risk_list = [0,1,2,3,4,5,6,7]
Risk_Description = ['Undefined', 'No', 'Low', 'Medium', 'High','Super','Extra High','Unacceptable']
#Creating spark dataframe from lists
df_gcob_static_RiskLevel = spark.createDataFrame(zip(Risk_list, Risk_Description), ['Id', 'Description'])
df_gcob_static_RiskLevel.createOrReplaceTempView('static_RiskLevel')

#Creating Static view for Risk Level
Risk_list = [0,1,2,3,4,9999]
Risk_Description = ['RiskNotAssigned','Unacceptable', 'High','Medium', 'Low', 'Unknown']
df_KN1_static_RiskLevel = spark.createDataFrame(zip(Risk_list, Risk_Description), ['Id', 'Description'])
df_KN1_static_RiskLevel.createOrReplaceTempView('kn1_static_RiskLevel')

#Creating Static view for KN1 Risk Level
Risk_list = [0,1,2,3,4]
Risk_Description = ['Incomplete', 'Low', 'Medium', 'High','Unacceptable']
#Creating spark dataframe from lists
df_gcob_RiskLevel = spark.createDataFrame(zip(Risk_list, Risk_Description), ['RiskModelRiskLevelId', 'DisplayName'])
df_gcob_RiskLevel.createOrReplaceTempView('RiskModelRiskLevel')

In [0]:
%sql
Create or replace temporary view Gcob_CaseClientDetails As
select * 
,Case
      when ClientType = 'Legal Entity' then concat('LEC_', GcobId)
      when ClientType in (
        'Natural Person',
        'Natural Person acting in a Professional Capacity (NPPC)'
      ) then concat('NP_NPPC_', GcobId)
    End as UniquePartyId
, MAX(CASE WHEN IsLatestApprovedVersionOfClient = true THEN CaseId END) OVER (PARTITION BY GcobId) AS LatestApprovedCaseId
from party_case_client_details
where CaseStatusName <> 'Cancelled'

######GCOB CASE INFORMATION

In [0]:
%sql
CREATE
OR REPLACE TEMPORARY VIEW GCOB_Cases AS
Select
  CONCAT('GCOB_',c.UniquePartyId) AS LocalSystemIdentifier,
  c.ClientId,
  c.CaseId,
  CASE
    WHEN c.ClientType like 'Legal%' THEN concat(c.SourceSystem, '_LEC_', c.CaseId)
    WHEN c.ClientType LIKE 'Natural%' THEN concat(c.SourceSystem, '_NP_NPPC_', c.CaseId)
  END as UniqueCaseId,
  c.LatestApprovedCaseId,
  c.ReviewTypeName,
  c.CaseStatusName,
  c.ReviewReason,
  CASE
    WHEN c.ReviewTypeName = 'Event Driven Review' THEN c.EdrReason
    WHEN c.ReviewTypeName = 'Amendment' THEN c.AmendmentReason
    WHEN c.ReviewTypeName = 'Change of Client Owner' THEN c.ClientOwnerChangeReason
    WHEN c.ReviewTypeName = 'Product Offboarding' THEN c.OffBoardingReason
    WHEN c.ReviewTypeName = 'Product Offboarding (Resume)' THEN c.OffBoardingReason
    WHEN c.ReviewTypeName = 'Tailored Event Assessment' THEN c.TEAReviewReasonDescription
    ELSE NULL
  END AS ReviewTypeReason,
  CASE
    WHEN c.ReviewTypeName = 'Event Driven Review' and c.EdrReason='Other' THEN c.EdrOtherReason
    WHEN c.ReviewTypeName = 'Tailored Event Assessment' and c.TEAReviewReasonDescription = 'Other' THEN c.TeaOtherReason
    ELSE NULL
  END AS ReviewTypeOtherReason,
  c.NextReviewDate,
  c.SubmitToClientCommittee,
  c.CaseDecisionMotivation,
  c.CurrentAssignee,
  c.InitiationInProgressAssignee,
  c.ReadyForKYCAssessmentDate,
  c.KYCAssessmentInProgressAssignee,
  c.LastKYCAnalyst,
  c.LastSentForKYCAssesment,
  c.DateSubmittedFor4EyeCheck,
  c.4EyeCheckReviewer,
  c.Last4EyeCheckReviewer,
  c.LastSentFor4EyeCheck,
  c.DateSubmittedForSignOff,
  c.ClientOwnerSignOffDate,
  c.LastProductOffboardingAnalyst,
  c.ValidatedRiskLevel,
  c.CaseCreationDate,
  c.ScheduledCompletionDate,
  c.CaseCompletedDate,
  c.FinalDecisionDate,
  c.RiskModelName,
  c.ModelCalculatedRiskLevel as CalculatedRiskLevel,
  c.ModelRecalculatedRiskLevel as ReCalculatedRiskLevel,
  c.RiskDeviationReason,
  null as DeactivationDate
from
  Gcob_CaseClientDetails c


######LEGACY2 CASE INFORMATION

In [0]:
%sql
CREATE OR REPLACE TEMPORARY VIEW NonSFDCID_Legacy2_Client AS
select * from Legacy2_case_client_details where isclient = 1 and GcobCaseId is null and Value is null and GcobId like 'RA:%';

In [0]:
%sql
CREATE OR REPLACE TEMPORARY VIEW Legacy2_client AS
select * from Legacy2_case_client_details
where ClientId not in (select ClientId from NonSFDCID_Legacy2_Client)

In [0]:
%sql
CREATE
OR REPLACE TEMPORARY VIEW Legacy2_Cases AS
Select
 CONCAT('LEGACY2_', CASE
    WHEN IsClient = 'true'
    and ClientTypeId = 1 THEN concat('LE_', c.GcobId)
    WHEN IsClient = 'true'
    and ClientTypeId in (2, 3) THEN concat('NP_NPPC_', c.GcobId)
    WHEN IsClient = 'false'
    and ClientTypeId = 1 THEN concat('RLE_', c.GcobId)
    WHEN IsClient = 'false'
    and ClientTypeId in (2, 3) THEN concat('RNP_', c.GcobId) END) as LocalSystemIdentifier,
  c.ClientId,
  null as CaseId,
  CASE
    WHEN c.IsClient = 'true'
    and c.ClientTypeId = 1 THEN concat(c.SourceSystem, '_LEC_', c.ClientId)
    WHEN c.IsClient = 'true'
    and c.ClientTypeId in (2, 3) THEN concat(c.SourceSystem, '_NP_NPPC_', c.ClientId)
    WHEN c.IsClient = 'false'
    and c.ClientTypeId = 1 THEN concat(c.SourceSystem, '_RLEP_', c.ClientId)
    WHEN c.IsClient = 'false'
    and c.ClientTypeId in (2, 3) THEN concat(c.SourceSystem, '_RNPP_', c.ClientId)
  END as UniqueCaseId,
  c.ClientId as LatestApprovedCaseId,
  c.ReviewTypeName,
  c.StatusTypeName as CaseStatusName,
  null as ReviewReason,
  CASE
    WHEN c.ReviewTypeName = 'Client Offboarding' THEN c.OffBoardingReasonDescription
    WHEN c.ReviewTypeName = 'Tailored Event Assessment' THEN c.TEAReviewReasonDescription
    ELSE NULL
  END AS ReviewTypeReason,
  CASE
    WHEN c.ReviewTypeName = 'Tailored Event Assessment' THEN c.TeaOtherReason
    ELSE NULL
  END AS ReviewTypeOtherReason,
  c.NextReviewDate,
  null as SubmitToClientCommittee,
  null as CaseDecisionMotivation,
  c.CurrentAssignee,
  c.InitiationInProgressAssignee,
  c.ReadyForKYCAssessmentDate,
  c.KYCAssessmentInProgressAssignee,
  c.LastKYCAnalyst,
  c.LastSentForKYCAssesment,
  c.DateSubmittedFor4EyeCheck,
  c.Eye4CheckReviewer as 4EyeCheckReviewer,
  c.Last4EyeCheckReviewer,
  c.LastSentFor4EyeCheck,
  c.DateSubmittedForSignOff,
  c.ClientOwnerSignOffDate,
  null as LastProductOffboardingAnalyst,
  r.ValidatedRiskLevel,
  c.CaseCreationDate,
  c.ScheduledCompletionDate,
  c.CaseCompletedDate,
  c.FinalDecisionDate,
  r.RiskModelName,
  r.CalculatedRiskLevel,
  r.ReCalculatedRiskLevel,
  r.RiskDeviationReason,
  null as DeactivationDate
from
  Legacy2_client c
  left join Legacy2_risk r on r.ClientId = c.ClientId
where
  c.IsClient = 'true'
  and c.ClientTypeId in (1, 2, 3)

######KN1 CASE INFORMATION

In [0]:
%sql
CREATE OR REPLACE TEMPORARY VIEW KN1_Cases AS
select 
  CONCAT('GIC_',gic.COD_INSTITUCIONAL) as LocalSystemIdentifier,
  gic.COD_INSTITUCIONAL as ClientID,
  cs.CaseID,
  CASE
    WHEN gic.SEQ_TIPO_PESSOA = 2 THEN concat('KN1_NP_NPPC_', cs.CaseId)
    ELSE concat('KN1_LEC_', cs.CaseId)
  END as UniqueCaseId,
  MAX(CASE WHEN cs.CaseStatus <> 'Cancelled' THEN cs.CaseID END) OVER (PARTITION BY gic.SEQ_PESSOA) AS LatestApprovedCaseId,
  cs.ReviewType as ReviewTypeName,
  cs.CaseStatus as CaseStatusName,
  acf.DE_ANALISE as ReviewReason,
  null as ReviewTypeReason,
  acf.DE_RESULTADO_SISTEMA as ReviewTypeOtherReason,
  cs.NextReviewDate,
  null as SubmitToClientCommittee,
  null as CaseDecisionMotivation,
  null as CurrentAssignee,
  null as InitiationInProgressAssignee,
  null as ReadyForKYCAssessmentDate,
  null as KYCAssessmentInProgressAssignee,
  null as LastKYCAnalyst,
  null as LastSentForKYCAssesment,
  null as DateSubmittedFor4EyeCheck,
  null as 4EyeCheckReviewer,
  null as Last4EyeCheckReviewer,
  null as LastSentFor4EyeCheck,
  kn_cont.DTHR_FIM as DateSubmittedForSignOff,
  null as ClientOwnerSignOffDate,
  null as LastProductOffboardingAnalyst,
  gicrs.GIC_Risk_Description as ValidatedRiskLevel,
  cs.CaseCreationDate,
  null as ScheduledCompletionDate,
  cs.CaseCompletedDate,
  cs.FinalDecisionDate,
  ri.RiskModelName,
  r.Description as CalculatedRiskLevel,
  rl.Description as RecalculatedRiskLevel,
  ri.RiskDeviationReason,
  gic.DTA_DESATIVACAO as DeactivationDate
from KN1Cases cs
  Inner join ADKYC_CONTRAPARTES kn_cont on cs.Caseid= kn_cont.ID_CONTRAPARTE
  Inner JOIN pessoa gic on cs.GicId = gic.COD_INSTITUCIONAL
  left join ADRBB_CONTRAPARTES_GRAM stat on trim(kn_cont.ID_CONTRAPARTE) = trim(stat.ID_CONTRAPARTE)
  Left join gic_static_Other_RiskLevel gicrs on kn_cont.CD_RISCO_ATRIBUIDO=gicrs.GIC_Risk_Id
  Left join  ADKYC_CONTRAPARTES_FASES acf on acf.ID_CONTRAPARTE=kn_cont.ID_CONTRAPARTE
  left join Party_RiskModelInstance ri on trim(stat.ID_GRAM_IDENTITY) = trim(ri.InstanceId)
  left join kn1_static_RiskLevel r on trim(kn_cont.CD_RISCO_MANUAL) = trim(r.Id)
  left join kn1_static_RiskLevel rl on trim(kn_cont.CD_RISCO_CALCULADO) = trim(rl.Id)
    and trim(ri.SourceSystemName) = 'Brazil - KN1'

MDM RANZ Cases

In [0]:
%sql
CREATE OR REPLACE TEMPORARY VIEW MDM_RANZ_Cases AS
select distinct
CONCAT('RANZ_',p.PKEY_SRC_OBJECT) LocalSystemIdentifier,
p.PKEY_SRC_OBJECT as ClientID,
dd.CASE_ID as CaseID,
null as UniqueCaseId, 
null as LatestApprovedCaseId,
null as ReviewTypeName,
'Completed' as CaseStatusName,
null as ReviewReason,
null as ReviewTypeReason,
null as ReviewTypeOtherReason,
dd.CDD_NEXT_REVIEW_DT as NextReviewDate,
null as SubmitToClientCommittee,
null as CaseDecisionMotivation,
c.CONTR_NAME as CurrentAssignee,
null as InitiationInProgressAssignee,
c.START_DT as ReadyForKYCAssessmentDate,
null as KYCAssessmentInProgressAssignee,
null as LastKYCAnalyst,
null as LastSentForKYCAssesment,
null as DateSubmittedFor4EyeCheck,
null as 4EyeCheckReviewer,
null as Last4EyeCheckReviewer,
null as LastSentFor4EyeCheck,
null as DateSubmittedForSignOff,
dd.CDD_SIGN_OFF_DT as ClientOwnerSignOffDate,
null as LastProductOffboardingAnalyst,
c.CDD_RISK_RATING_DESC as ValidatedRiskLevel,
dd.START_DT as CaseCreationDate,
dd.CDD_COMPLETION_DT as ScheduledCompletionDate,
dd.END_DT as CaseCompletedDate,
dd.CDD_STRATEGIC_REVIEW_DT as FinalDecisionDate,
 d.RiskModelName,
 e.Description as CalculatedRiskLevel,
 f.Description as RecalculatedRiskLevel,
 d.RiskDeviationReason,
null as DeactivationDate
FROM c_b_party as p
JOIN c_b_contr_rol_party as crp
  ON crp.FK_PARTY_ID = p.ROWID_XREF
JOIN c_b_contract as c
  ON c.CONTR_ID = crp.FK_CONTR_ID
LEFT JOIN c_b_due_diligence as dd
  ON dd.FK_CONTR_ID = c.CONTR_ID
join c_lkp_cdd_risk_rating c
on c.CDD_RISK_RATING_CODE = dd.CDD_RISK_RTG_CD
left join Party_RiskModelInstance d
on d.InstanceId = dd.GRAM_INSTANCE_ID
 and d.SourceSystemName = 'RANZ - PegaCdd'
 left join static_RiskLevel e
 on d.OverallCalculatedRiskLevelId = e.Id
  left join static_RiskLevel f
 on d.OverallRecalculatedRiskLevelId = f.Id

In [0]:
%skip
select *
FROM c_b_party as p
JOIN c_b_contr_rol_party as crp
    ON crp.FK_PARTY_ID = p.ROWID_XREF
JOIN c_b_contract as c
    ON c.CONTR_ID = crp.FK_CONTR_ID
LEFT JOIN c_b_due_diligence as dd
    ON dd.FK_CONTR_ID = c.CONTR_ID;

##### CREATING A DATAFRAME BY COMBINING ALL THE CASE INFORMATION FROM DIFFERENT SOURCESYSTEMS AND LOADING THE FINAL DATAOBJECT TO SA RADAR STORAGE ACCOUNT


In [0]:
%sql
CREATE OR REPLACE TEMPORARY VIEW CDD_Cases AS
with combinedData as(
select * FROM GCOB_Cases
UNION
select * from Legacy2_Cases
UNION
select * from KN1_Cases
UNION
select * from MDM_RANZ_Cases
)

Select *, CASE
    WHEN CaseStatusName in("Approved","Completed") Then "Completed"
    WHEN CaseStatusName in("Cancelled") Then "Cancelled"
    WHEN CaseStatusName in("Migrated") Then "Migrated"
    ELSE "InProgress"
  END as StandardCaseStatus from combinedData

In [0]:
%sql
CREATE OR REPLACE TEMPORARY VIEW CDD_Cases_final AS
(
  select 
  LocalSystemIdentifier,
  CASE 
    WHEN ClientId RLIKE '^[0-9]+$' THEN CAST(ClientId AS INT)
    ELSE NULL
  END AS ClientId,
  CAST(ClientID AS STRING) AS ClientId_string,
  CASE 
    WHEN CaseId RLIKE '^[0-9]+$' THEN CAST(CaseId AS INT)
    ELSE NULL
  END AS CaseId,
  CAST(CaseId AS STRING) AS CaseId_string,
  UniqueCaseId,
  LatestApprovedCaseId,
  CASE
    WHEN CaseId IS NOT NULL AND LatestApprovedCaseId IS NOT NULL AND CaseId = LatestApprovedCaseId
    THEN TRUE
    ELSE FALSE
  END AS IsLatestApprovedCase,
  ReviewTypeName,
  CaseStatusName,
  array_sort(collect_set(nullif(trim(ReviewReason), ''))) AS ReviewReason,
  ReviewTypeReason,
  array_sort(collect_set(nullif(trim(ReviewTypeOtherReason), ''))) AS ReviewTypeOtherReason,
  NextReviewDate,
  SubmitToClientCommittee,
  CaseDecisionMotivation,
  CurrentAssignee,
  InitiationInProgressAssignee,
  ReadyForKYCAssessmentDate,
  KYCAssessmentInProgressAssignee,
  LastKYCAnalyst,
  LastSentForKYCAssesment,
  DateSubmittedFor4EyeCheck,
  4EyeCheckReviewer,
  Last4EyeCheckReviewer,
  LastSentFor4EyeCheck,
  DateSubmittedForSignOff,
  ClientOwnerSignOffDate,
  LastProductOffboardingAnalyst,
  ValidatedRiskLevel,
  CaseCreationDate,
  ScheduledCompletionDate,
  CaseCompletedDate,
  FinalDecisionDate,
  RiskModelName,
  CalculatedRiskLevel,
  ReCalculatedRiskLevel,
  RiskDeviationReason,
  StandardCaseStatus,
  DeactivationDate
from CDD_Cases
GROUP BY
  LocalSystemIdentifier,
  ClientID,
  CaseID,
  UniqueCaseId,
  LatestApprovedCaseId,
  IsLatestApprovedCase,
  ReviewTypeName,
  CaseStatusName,
  ReviewTypeReason,
  NextReviewDate,
  SubmitToClientCommittee,
  CaseDecisionMotivation,
  CurrentAssignee,
  InitiationInProgressAssignee,
  ReadyForKYCAssessmentDate,
  KYCAssessmentInProgressAssignee,
  LastKYCAnalyst,
  LastSentForKYCAssesment,
  DateSubmittedFor4EyeCheck,
  `4EyeCheckReviewer`,
  Last4EyeCheckReviewer,
  LastSentFor4EyeCheck,
  DateSubmittedForSignOff,
  ClientOwnerSignOffDate,
  LastProductOffboardingAnalyst,
  ValidatedRiskLevel,
  CaseCreationDate,
  ScheduledCompletionDate,
  CaseCompletedDate,
  FinalDecisionDate,
  RiskModelName,
  CalculatedRiskLevel,
  ReCalculatedRiskLevel,
  RiskDeviationReason,
  StandardCaseStatus,
  DeactivationDate
);


In [0]:
df_party_CDDcase=spark.table('CDD_Cases_final')

In [0]:
df_final_party_CDDcase=add_party_identifier(df_party_CDDcase,df_Party_SystemIdentifier)

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window


def resolve_latest_approved_case_per_party(df):
    """
    Resolves duplicate IsLatestApprovedCase entries per PartyIdentifier.
    The logic is applied ONLY when a PartyIdentifier has
    more than one IsLatestApprovedCase = TRUE.

    Preference rules:
      1. Source system priority:
         GCOB > KN1 > LEGACY2 > RANZ
      2. If same system, latest CaseCompletedDate gets priority
    """

    #Identifying rows with more than 1 IsLatestApprovedCase

    duplicate_parties = (
        df.filter(F.col("IsLatestApprovedCase") == True)
          .groupBy("PartyIdentifier")
          .count()
          .filter(F.col("count") > 1)
          .select("PartyIdentifier")
    )

    # Add a helper column (NeedsFix) to flag PartyIdentifiers that have more than one latest-approved case.
    #Coalesce is used to replace nulls with False, so every row has a clear True/False indicator. 

    df = (
        df.join(
            duplicate_parties.withColumn("NeedsFix", F.lit(True)),
            on="PartyIdentifier",
            how="left"
        )
        .withColumn("NeedsFix", F.coalesce(F.col("NeedsFix"), F.lit(False)))
    )

    #Adding SourceSystem column which extracts the system name from UniqueCaseId column only for required rows

    df = df.withColumn(
        "SourceSystem",
        F.when(F.col("NeedsFix") == False, None)
         .when(F.col("UniqueCaseId").startswith("GCOB"), "GCOB")
         .when(F.col("UniqueCaseId").startswith("KN1"), "KN1")
         .when(F.col("UniqueCaseId").startswith("Legacy2"), "LEGACY2")
         .when(F.col("UniqueCaseId").startswith("RANZ"), "RANZ")
         .otherwise("UNKNOWN")
    )

    #Assigning Priority GCOB > KN1 > LEGACY2 > RANZ

    df = df.withColumn(
        "SystemPriority",
        F.when(F.col("NeedsFix") == False, None)
         .when(F.col("SourceSystem") == "GCOB", 1)
         .when(F.col("SourceSystem") == "KN1", 2)
         .when(F.col("SourceSystem") == "LEGACY2", 3)
         .when(F.col("SourceSystem") == "RANZ", 4)
         .otherwise(99)
    )

    #Defining a Window function which ranks cases for each PartyIdentifier. Case ordering is first by System Priority (Gcob gets ranked higher than Kn1) and then by the most recent CaseCompletedDate.

    window_spec = Window.partitionBy("PartyIdentifier").orderBy(
        F.col("SystemPriority").asc(),
        F.col("CaseCompletedDate").desc_nulls_last()
    )

    #Apply row number only to rows that need fixing and IsLatestApprovedCase =True. 
    #LatestRank helper column helps decide which row gets final priority. 
      
    df = df.withColumn(
        "LatestRank",
        F.when(
            (F.col("NeedsFix") == True) &
            (F.col("IsLatestApprovedCase") == True),
            F.row_number().over(window_spec)
        )
    )

    #Helper Column FinalLatestApprovedCaseId contains CaseId for cases ranked as the final LatestApprovedCase

    final_latest_approved_cases = (
        df.filter(F.col("LatestRank") == 1)
          .select(
              "PartyIdentifier",
              F.col("CaseId").alias("FinalLatestApprovedCaseId")
          )
    )

    #Updating LatestApprovedCaseId to show only one latest approved case for each PartyIdentifier. 
    df = (
        df.join(final_latest_approved_cases, on="PartyIdentifier", how="left")
        .withColumn(
            "LatestApprovedCaseId",
            F.when(
                F.col("NeedsFix") == True,
                F.col("FinalLatestApprovedCaseId")
            ).otherwise(F.col("LatestApprovedCaseId"))
        )
    )

    #Re-deriving IsLatestApprovedCase flag. 
    df = df.withColumn(
        "IsLatestApprovedCase",
        F.when(
            (F.col("CaseId").isNotNull()) &
            (F.col("LatestApprovedCaseId").isNotNull()) &
            (F.col("CaseId") == F.col("LatestApprovedCaseId")),
            True
        ).otherwise(False)
    )
 
    #Dropping Helper Columns
    df = df.drop(
        "NeedsFix",
        "SourceSystem",
        "SystemPriority",
        "LatestRank",
        "FinalLatestApprovedCaseId"
    )

    return df

In [0]:
#Using function to resolve duplicate values of LatestApprovedCaseId
df_final_party_CDDcase = resolve_latest_approved_case_per_party(df_final_party_CDDcase)

In [0]:
if RunType == "historical":
    save_to_saradar_storage_account(df_final_party_CDDcase, party_CDDcase_dataobject, radar_datamodel_version_number, environment,Load_Date)
else:
    save_to_saradar_storage_account(df_final_party_CDDcase, party_CDDcase_dataobject, radar_datamodel_version_number, environment)